# SetFit Methodology Experiment

The goal of this notebook is to test the methodology from `Efficient Few-Shot Learning Without Prompts` on the router task.

The paper proposes SetFit, a prompt-free few-shot classification method. Instead of writing prompts or verbalizers, SetFit uses a small number of labeled examples to fine-tune a SentenceTransformer with contrastive sentence pairs. The fine-tuned encoder is then used to produce embeddings for a lightweight classification head.

In this notebook we adapt that idea to the Law/Finance/Health router:
- sample only a few labeled examples per in-domain class
- generate positive and negative text pairs from those examples
- fine-tune a SentenceTransformer with a contrastive objective
- train a logistic-regression classification head on the resulting embeddings
- add a confidence threshold so uncertain queries can be rejected as OOD

Sources: arXiv paper https://arxiv.org/abs/2209.11055 and the Hugging Face SetFit overview https://huggingface.co/blog/setfit.

## 1. Environment

The first cell prepares the notebook environment, points Python at the local `src` package, and enables the project-local Hugging Face cache under `data/cache`.

In [ ]:
from __future__ import annotations

import logging
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from router.cache import configure_project_cache, sentence_transformers_cache_dir

DATA_CACHE_DIR = configure_project_cache()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

{"project_root": PROJECT_ROOT, "data_cache_dir": DATA_CACHE_DIR}

The output confirms that the notebook is using the repository as the working project and that downloaded models or datasets will be reused from the local cache on later runs.

## 2. Methodology Summary

SetFit is a two-stage method:
- first, fine-tune a SentenceTransformer body on positive and negative pairs built from a few labeled examples
- second, freeze/use that body to encode texts and train a small classification head

Positive pairs contain two examples from the same class. Negative pairs contain examples from different classes. This turns a tiny labeled dataset into a larger set of pairwise training examples without using prompts.

## 3. Configuration

The default configuration follows the common SetFit few-shot setup with `8` labeled examples per class and one contrastive fine-tuning epoch. For faster smoke tests, reduce `N_SHOTS`, `NUM_ITERATIONS`, or switch to `sentence-transformers/all-MiniLM-L6-v2`.

In [ ]:
SEED = 42

# The SetFit blog uses paraphrase-mpnet-base-v2 as a strong default.
EMBEDDING_MODEL = "sentence-transformers/paraphrase-mpnet-base-v2"
# EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # faster local smoke test

N_SHOTS = 8
NUM_ITERATIONS = 20
NUM_EPOCHS = 1
CONTRASTIVE_BATCH_SIZE = 16
ENCODE_BATCH_SIZE = 64

MAX_VALID_SAMPLES = 3000
MAX_OOD_SAMPLES = 1000

# Keep this True to test the SetFit methodology. Set False to compare against a frozen encoder.
RUN_CONTRASTIVE_FINE_TUNING = True

SETFIT_MODEL_DIR = PROJECT_ROOT / "artifacts" / "setfit_router"

The most important experimental variable is `N_SHOTS`. With `8`, the classifier sees only eight law, eight finance, and eight health examples before evaluation.

## 4. Load Router Data

The SetFit classifier is trained only on the in-domain classes. OOD data is loaded separately so we can later inspect whether a confidence threshold can reject unsupported queries.

In [ ]:
from router.data import (
    DatasetSplit,
    load_builtin_ood_validation_dataset,
    load_gqr_ood_test_dataset,
    load_gqr_train_dataset,
)
from router.labels import ID_LABELS, LABEL_TO_DOMAIN, OOD


def split_to_frame(split: DatasetSplit, split_name: str) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "text": split.texts,
            "label": split.labels,
            "domain": [LABEL_TO_DOMAIN[label] for label in split.labels],
            "split": split_name,
        }
    )


def load_ood_split(max_samples: int) -> DatasetSplit:
    try:
        return load_gqr_ood_test_dataset().limit(max_samples)
    except Exception as exc:
        print(f"Falling back to built-in OOD examples: {type(exc).__name__}: {exc}")
        return load_builtin_ood_validation_dataset(max_samples=max_samples)


load_start = time.perf_counter()
train_split, valid_split = load_gqr_train_dataset()
valid_split = valid_split.limit(MAX_VALID_SAMPLES)
ood_split = load_ood_split(MAX_OOD_SAMPLES)
data_loading_seconds = time.perf_counter() - load_start

train_df = split_to_frame(train_split, "train")
valid_df = split_to_frame(valid_split, "valid")
ood_df = split_to_frame(ood_split, "ood")

pd.concat([train_df, valid_df, ood_df]).groupby(["split", "domain"]).size().unstack(fill_value=0)

The table shows the available data before few-shot sampling. The training pool can be large, but the SetFit experiment below intentionally keeps only a tiny labeled subset.

## 5. Build The Few-Shot Training Set

We sample exactly `N_SHOTS` examples from each in-domain class. This creates a small labeled set that simulates the label-scarce setting studied in the paper.

In [ ]:
def sample_few_shot_examples(
    frame: pd.DataFrame,
    n_shots: int,
    seed: int,
) -> pd.DataFrame:
    sampled_frames = []
    for label in ID_LABELS:
        class_frame = frame[frame["label"] == label]
        if len(class_frame) < n_shots:
            domain = LABEL_TO_DOMAIN[label]
            raise ValueError(
                f"Need {n_shots} examples for {domain}, found {len(class_frame)}"
            )
        sampled_frames.append(class_frame.sample(n=n_shots, random_state=seed + label))
    return pd.concat(sampled_frames).sample(frac=1, random_state=seed).reset_index(drop=True)


few_shot_df = sample_few_shot_examples(train_df, N_SHOTS, SEED)
few_shot_df.groupby("domain").size().to_frame("few_shot_examples")

The sampled set is deliberately small. The contrastive step will expand this into many training pairs, but all pairs still come from these few original labeled examples.

## 6. Generate Contrastive Pairs

The pair generator creates both positive and negative examples. A positive pair receives label `1.0`, and a negative pair receives label `0.0`. This mirrors the contrastive Siamese training idea used by SetFit.

In [ ]:
from sentence_transformers import InputExample


def build_contrastive_pairs(
    frame: pd.DataFrame,
    num_iterations: int,
    seed: int,
) -> list[InputExample]:
    rng = np.random.default_rng(seed)
    by_label = {
        label: frame[frame["label"] == label]["text"].tolist()
        for label in sorted(frame["label"].unique())
    }
    labels = list(by_label)
    examples: list[InputExample] = []

    for _ in range(num_iterations):
        for label, texts in by_label.items():
            first, second = rng.choice(texts, size=2, replace=False)
            examples.append(InputExample(texts=[first, second], label=1.0))

            other_label = int(rng.choice([candidate for candidate in labels if candidate != label]))
            negative = rng.choice(by_label[other_label])
            examples.append(InputExample(texts=[first, negative], label=0.0))

    rng.shuffle(examples)
    return examples


pair_examples = build_contrastive_pairs(few_shot_df, NUM_ITERATIONS, SEED)
pd.DataFrame(
    [
        {"pair_label": example.label, "text_a": example.texts[0], "text_b": example.texts[1]}
        for example in pair_examples[:6]
    ]
)

The pair count grows with `NUM_ITERATIONS`, not with the original dataset size. This is the main trick that lets SetFit get useful supervision from very few labeled examples.

## 7. Fine-Tune The SentenceTransformer Body

The SentenceTransformer is trained on the generated pairs with a cosine-similarity loss. If `RUN_CONTRASTIVE_FINE_TUNING` is set to `False`, this cell skips fine-tuning and the rest of the notebook becomes a frozen-embedding baseline.

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import losses
from torch.utils.data import DataLoader

body_start = time.perf_counter()
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    cache_folder=str(sentence_transformers_cache_dir()),
)

pair_loader = DataLoader(
    pair_examples,
    shuffle=True,
    batch_size=CONTRASTIVE_BATCH_SIZE,
)
train_loss = losses.CosineSimilarityLoss(embedding_model)

if RUN_CONTRASTIVE_FINE_TUNING:
    warmup_steps = max(1, int(len(pair_loader) * NUM_EPOCHS * 0.1))
    embedding_model.fit(
        train_objectives=[(pair_loader, train_loss)],
        epochs=NUM_EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
    )
else:
    print("Skipping contrastive fine-tuning; using the frozen encoder.")

body_training_seconds = time.perf_counter() - body_start
{"pairs": len(pair_examples), "body_training_seconds": round(body_training_seconds, 2)}

The encoder now has task-specific contrastive tuning from the few-shot examples. The next step follows the paper's second stage: encode labeled texts and train a small classifier head.

## 8. Train The Classification Head

The classification head is a logistic-regression model trained on embeddings from the few-shot examples. This keeps the head simple and makes the effect of the SentenceTransformer body easier to inspect.

In [ ]:
from sklearn.linear_model import LogisticRegression


def encode_texts(model: SentenceTransformer, texts: list[str]) -> np.ndarray:
    return model.encode(
        texts,
        batch_size=ENCODE_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=len(texts) >= 512,
    )


head_start = time.perf_counter()
few_shot_embeddings = encode_texts(embedding_model, few_shot_df["text"].tolist())

classifier = LogisticRegression(
    C=2.0,
    class_weight="balanced",
    max_iter=1000,
    random_state=SEED,
)
classifier.fit(few_shot_embeddings, few_shot_df["label"].tolist())
head_training_seconds = time.perf_counter() - head_start

{"few_shot_rows": len(few_shot_df), "head_training_seconds": round(head_training_seconds, 2)}

The classifier has only seen the few-shot examples. All larger validation and OOD data below is used to evaluate behavior, not to train the classification head.

## 9. Evaluate In-Domain Accuracy

We first evaluate the SetFit-style classifier as a normal three-way domain classifier, without OOD thresholding. This measures how well the few-shot method routes supported queries.

In [ ]:
from router.metrics import accuracy, confusion_counts, evaluate_router

eval_start = time.perf_counter()
valid_embeddings = encode_texts(embedding_model, valid_df["text"].tolist())
valid_probabilities = classifier.predict_proba(valid_embeddings)
valid_class_indices = np.argmax(valid_probabilities, axis=1)
valid_predictions = [int(classifier.classes_[index]) for index in valid_class_indices]
valid_confidences = valid_probabilities.max(axis=1)

id_accuracy = accuracy(valid_df["label"].tolist(), valid_predictions)
eval_id_seconds = time.perf_counter() - eval_start

pd.DataFrame(
    [
        {
            "id_accuracy": id_accuracy,
            "valid_rows": len(valid_df),
            "mean_confidence": float(np.mean(valid_confidences)),
            "eval_id_seconds": round(eval_id_seconds, 2),
        }
    ]
)

This table evaluates only supported domains. A strong result here means the few-shot classifier can separate law, finance, and health queries, but it does not yet show whether unsupported queries are rejected.

## 10. Add OOD Rejection By Thresholding

The original SetFit paper focuses on few-shot classification. For this router project, we add a simple confidence threshold: if the classifier's best probability is below the threshold, the prediction becomes OOD.

In [ ]:
ood_embeddings = encode_texts(embedding_model, ood_df["text"].tolist())
report_df = pd.concat([valid_df, ood_df], ignore_index=True)
report_embeddings = np.vstack([valid_embeddings, ood_embeddings])
report_probabilities = classifier.predict_proba(report_embeddings)
report_best_indices = np.argmax(report_probabilities, axis=1)
report_best_labels = [int(classifier.classes_[index]) for index in report_best_indices]
report_confidences = report_probabilities.max(axis=1)

threshold_rows = []
for threshold in np.linspace(0.0, 1.0, 101):
    routed_predictions = [
        label if confidence >= threshold else OOD
        for label, confidence in zip(report_best_labels, report_confidences)
    ]
    scores = evaluate_router(report_df["label"].tolist(), routed_predictions)
    threshold_rows.append(
        {
            "threshold": threshold,
            "id_accuracy": scores.id_accuracy,
            "ood_accuracy": scores.ood_accuracy,
            "gqr_score": scores.gqr_score,
        }
    )

threshold_results = pd.DataFrame(threshold_rows)
best_threshold_row = threshold_results.sort_values("gqr_score", ascending=False).iloc[0]
BEST_THRESHOLD = float(best_threshold_row["threshold"])

threshold_results.sort_values("gqr_score", ascending=False).head(10)

The threshold sweep shows the trade-off between routing valid in-domain examples and rejecting OOD examples. Higher thresholds usually improve OOD rejection but may reject too many valid queries.

In [ ]:
best_routed_predictions = [
    label if confidence >= BEST_THRESHOLD else OOD
    for label, confidence in zip(report_best_labels, report_confidences)
]

confusion_rows = [
    {
        "true_label": true,
        "true_domain": LABEL_TO_DOMAIN[true],
        "predicted_label": pred,
        "predicted_domain": LABEL_TO_DOMAIN[pred],
        "count": count,
    }
    for (true, pred), count in sorted(
        confusion_counts(report_df["label"].tolist(), best_routed_predictions).items()
    )
]

pd.DataFrame(confusion_rows)

The confusion table makes the threshold behavior visible. In-domain rows predicted as `ood` are over-rejections, while OOD rows predicted as law, finance, or health are missed rejections.

## 11. Inspect Example Predictions

Manual examples help check whether the learned router behaves sensibly beyond aggregate scores.

In [ ]:
sample_queries = [
    "Can my landlord keep my deposit after I move out?",
    "What are the risks of buying index funds right now?",
    "What are common symptoms of low blood pressure?",
    "How do I tune a guitar?",
]

sample_embeddings = encode_texts(embedding_model, sample_queries)
sample_probabilities = classifier.predict_proba(sample_embeddings)
sample_best_indices = np.argmax(sample_probabilities, axis=1)
sample_best_labels = [int(classifier.classes_[index]) for index in sample_best_indices]
sample_confidences = sample_probabilities.max(axis=1)
sample_routed_labels = [
    label if confidence >= BEST_THRESHOLD else OOD
    for label, confidence in zip(sample_best_labels, sample_confidences)
]

pd.DataFrame(
    {
        "query": query,
        "best_domain_before_threshold": LABEL_TO_DOMAIN[best_label],
        "confidence": round(float(confidence), 4),
        "final_domain": LABEL_TO_DOMAIN[routed_label],
    }
    for query, best_label, confidence, routed_label in zip(
        sample_queries,
        sample_best_labels,
        sample_confidences,
        sample_routed_labels,
    )
)

The `best_domain_before_threshold` column shows what the classifier would choose if forced to pick a supported domain. The `final_domain` column shows the router decision after thresholding.

## 12. Save Artifacts

The saved output contains the fine-tuned SentenceTransformer body, the classifier head, and the threshold found in the sweep. This keeps the SetFit-style experiment separate from the baseline router artifacts.

In [ ]:
SETFIT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
embedding_model.save(str(SETFIT_MODEL_DIR / "embedding_model"))
joblib.dump(
    {
        "embedding_model": str(SETFIT_MODEL_DIR / "embedding_model"),
        "classifier": classifier,
        "threshold": BEST_THRESHOLD,
        "label_to_domain": LABEL_TO_DOMAIN,
        "n_shots": N_SHOTS,
        "num_iterations": NUM_ITERATIONS,
        "source_paper": "https://arxiv.org/abs/2209.11055",
    },
    SETFIT_MODEL_DIR / "setfit_router.joblib",
)

{"saved_to": str(SETFIT_MODEL_DIR), "threshold": BEST_THRESHOLD}

The artifacts are stored under `artifacts/setfit_router`, so they do not overwrite the earlier embedding-router baseline.

## 13. Runtime Summary

The paper emphasizes efficiency, so we keep a simple runtime table for data loading, contrastive body training, classifier-head training, and in-domain evaluation.

In [ ]:
runtime_rows = [
    {"stage": "data_loading", "seconds": data_loading_seconds},
    {"stage": "contrastive_body_training", "seconds": body_training_seconds},
    {"stage": "classifier_head_training", "seconds": head_training_seconds},
    {"stage": "id_evaluation", "seconds": eval_id_seconds},
]

pd.DataFrame(runtime_rows).assign(seconds=lambda frame: frame["seconds"].round(2))

## 14. Conclusion

This notebook tests the central SetFit idea from the paper in the router setting: use only a few labeled examples, expand them into contrastive pairs, fine-tune a SentenceTransformer, and train a lightweight classifier head.

The extra OOD thresholding section is specific to this project. It is not the main contribution of the SetFit paper, but it lets the prompt-free few-shot classifier behave like a guarded query router.

Useful follow-up experiments:
- compare `paraphrase-mpnet-base-v2` against `all-MiniLM-L6-v2`
- repeat the run with `N_SHOTS = 4`, `8`, and `16`
- tune the threshold on a held-out calibration split instead of the reporting split
- compare this notebook against `router_baseline.ipynb` using the same validation and OOD data